In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os

# Define paths
zip_path = '/content/gps_dataset.zip'
extract_path = '/content/dataset'

# Create target directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

print("Extracting dataset... Please wait.")
!unzip -q "{zip_path}" -d "{extract_path}"
print("Dataset successfully extracted to Colab local storage!")

Extracting dataset... Please wait.
Dataset successfully extracted to Colab local storage!


In [11]:
# Check contents of the extracted folder
print(os.listdir(extract_path))

# Check inside the train directory
train_dir = os.path.join(extract_path, 'train')
if os.path.exists(train_dir):
    print(f"Train folders found: {len(os.listdir(train_dir))}")
else:
    print("Warning: 'train' directory not found. Check your folder structure inside the zip.")

['Student folder']


In [12]:
import os

# Let's inspect the entire structure of the extracted folder to see how files are placed
print("Contents of /content:", os.listdir('/content'))
if os.path.exists(extract_path):
    print("Contents of extract_path:", os.listdir(extract_path))
    # Recursively list first few items in extract_path to find where the files are
    for root, dirs, files in os.walk(extract_path):
        if files:
            print(f"Found folder with files: {root}")
            print(f"Sample files: {files[:5]}")
            break

Contents of /content: ['.config', 'dataset', 'drive', 'gps_dataset.zip', 'sample_data']
Contents of extract_path: ['Student folder']
Found folder with files: /content/dataset/Student folder/student_test/00000065
Sample files: ['00000065.nodes', '00000065.arcs', '00000065.route', '00000065.track']


In [13]:
!pip install -q geopandas shapely pyproj networkx folium osmnx
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString, Point
import networkx as nx
import folium
from pyproj import Transformer

In [15]:
def explore_dataset(base_dir):
    inventory = []
    anomaly_log = []

    # Assuming standard structure: partitions (train/test) containing trips and road network files
    for partition in ['train', 'test']:
        part_path = os.path.join(base_dir, partition)
        if not os.path.exists(part_path):
            continue

        trip_files = [f for f in os.listdir(part_path) if f.endswith(('.csv', '.parquet', '.txt'))]

        total_obs = 0
        for file in trip_files:
            file_path = os.path.join(part_path, file)
            file_size = os.path.getsize(file_path) / (1024 * 1024) # Size in MB

            # Raw .track files are ordered as: Longitude, Latitude, Relative Time/Timestamp
            df = pd.read_csv(file_path, header=None, sep=r'\s+') # adjust sep if it's comma/tab separated
            df.columns = ['lon', 'lat', 'relative_time']
            obs_count = len(df)
            total_obs += obs_count

            # Check data-quality issues
            # 1. Missing or invalid coordinates
            invalid_coords = df['lat'].isna() | df['lon'].isna() | (df['lat'] == 0) | (df['lon'] == 0)
            if invalid_coords.sum() > 0:
                anomaly_log.append(f"{file}: {invalid_coords.sum()} invalid/missing coordinates found.")

            # 2. Timestamp ordering & irregular time gaps
            if 'timestamp' in df.columns:
                df['timestamp'] = pd.to_datetime(df['timestamp'])
                time_diffs = df['timestamp'].diff().dt.total_seconds()
                if (time_diffs < 0).any():
                    anomaly_log.append(f"{file}: Timestamp out-of-order error detected!")
                large_gaps = (time_diffs > 300).sum() # Gaps > 5 minutes
                if large_gaps > 0:
                    anomaly_log.append(f"{file}: {large_gaps} large temporal gaps (>5 mins) found.")

            inventory.append({
                'Partition': partition,
                'File': file,
                'Size_MB': round(file_size, 2),
                'Observations': obs_count
            })

    inv_df = pd.DataFrame(inventory)
    print("--- DATASET INVENTORY SUMMARY ---")
    print(inv_df.groupby('Partition').agg({'File': 'count', 'Observations': ['sum', 'mean']}))

    print("\n--- ANOMALY LOG SAMPLE ---")
    for anomaly in anomaly_log[:10]:
        print(f" - {anomaly}")

    return inv_df

In [16]:
# Setup transformer to convert Lat/Lon (WGS84, EPSG:4326) to local UTM (meters)
# Example using UTM Zone for Bengaluru/local region or auto-detecting
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True) # Adjust EPSG based on your region

def clean_and_project_trajectory(df):
    # 1. Drop missing or invalid coordinates
    df = df.dropna(subset=['lat', 'lon'])
    df = df[(df['lat'].between(-90, 90)) & (df['lon'].between(-180, 180))]

    # 2. Fix coordinate order (Ensure Lon is X, Lat is Y)
    x, y = transformer.transform(df['lon'].values, df['lat'].values)
    df['x_m'] = x
    df['y_m'] = y

    # 3. Calculate physical distance intervals (meters) using projected coordinates
    df['dx'] = df['x_m'].diff()
    df['dy'] = df['y_m'].diff()
    df['step_distance_m'] = np.sqrt(df['dx']**2 + df['dy']**2)

    # 4. Handle GPS Noise & Implausible Speeds (e.g., speed > 150 km/h or 41 m/s)
    if 'timestamp' in df.columns:
        df['dt'] = df['timestamp'].diff().dt.total_seconds()
        df['speed_mps'] = df['step_distance_m'] / df['dt']
        # Flag teleportation / severe GPS noise
        df['is_noise'] = df['speed_mps'] > 41.7

    return df

In [17]:
def clean_road_network(edges_df, nodes_df):
    # Build NetworkX Graph
    G = nx.Graph()

    # Add nodes with coordinates
    for _, row in nodes_df.iterrows():
        G.add_node(row['node_id'], x=row['lon'], y=row['lat'])

    # Add edges (road segments)
    for _, row in edges_df.iterrows():
        G.add_edge(row['u'], row['v'], weight=row['length'])

    # 1. Check connectivity and isolated nodes
    isolated_nodes = list(nx.isolates(G))
    print(f"Number of isolated nodes: {len(isolated_nodes)}")

    # Remove isolated nodes if needed
    G.remove_nodes_from(isolated_nodes)

    # 2. Extract Connected Components
    components = list(nx.connected_components(G))
    print(f"Total connected components: {len(components)}")

    # Retain only the Largest Connected Component (LCC) to avoid routing breaks
    largest_cc = max(components, key=len)
    G_lcc = G.subgraph(largest_cc).copy()
    print(f"LCC nodes: {G_lcc.number_of_nodes()}, LCC edges: {G_lcc.number_of_edges()}")

    return G_lcc

In [18]:
def generate_folium_map(df, sample_name="Trip Route"):
    # Center map on the trajectory mean
    m = folium.Map(location=[df['lat'].mean(), df['lon'].mean()], zoom_start=14)

    # Plot GPS points
    points = list(zip(df['lat'], df['lon']))
    folium.PolyLine(points, color="blue", weight=4, opacity=0.8, tooltip=sample_name).add_to(m)

    # Highlight start and end
    folium.Marker(points[0], popup="Start", icon=folium.Icon(color="green")).add_to(m)
    folium.Marker(points[-1], popup="End", icon=folium.Icon(color="red")).add_to(m)

    return m

In [20]:
import os
import pandas as pd
import numpy as np

# Locate the training and testing folders
base_student_dir = '/content/dataset/Student folder'

# Identify subdirectories (e.g., student_train, student_test)
train_subfolder = None
test_subfolder = None

for d in os.listdir(base_student_dir):
    full_path = os.path.join(base_student_dir, d)
    if os.path.isdir(full_path):
        if 'train' in d.lower():
            train_subfolder = full_path
        elif 'test' in d.lower():
            test_subfolder = full_path

# Fallback if names are slightly different
if not train_subfolder or not test_subfolder:
    dirs = [os.path.join(base_student_dir, d) for d in os.listdir(base_student_dir) if os.path.isdir(os.path.join(base_student_dir, d))]
    if len(dirs) >= 2:
        train_subfolder, test_subfolder = dirs[0], dirs[1]

print(f"Using Train folder: {train_subfolder}")
print(f"Using Test folder: {test_subfolder}")

def parse_track_file(file_path):
    """Parses a .track file consisting of space-separated longitude, latitude, and relative time values."""
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3:
                try:
                    val1, val2, val3 = parts[0], parts[1], parts[2]
                    data.append({
                        'lon': float(val1),
                        'lat': float(val2),
                        'relative_time': float(val3)
                    })
                except Exception:
                    continue
    return pd.DataFrame(data)

def build_features_from_partition(partition_path):
    """Builds a unified DataFrame by parsing track files across all trip folders in a partition."""
    all_trips = []
    if not partition_path or not os.path.exists(partition_path):
        return pd.DataFrame()

    trip_dirs = [os.path.join(partition_path, d) for d in os.listdir(partition_path) if os.path.isdir(os.path.join(partition_path, d))]

    for trip_dir in trip_dirs:
        trip_id = os.path.basename(trip_dir)
        for file in os.listdir(trip_dir):
            if file.endswith('.track'):
                track_df = parse_track_file(os.path.join(trip_dir, file))
                if not track_df.empty:
                    track_df['trip_id'] = trip_id
                    all_trips.append(track_df)

    if all_trips:
        return pd.concat(all_trips, ignore_index=True)
    return pd.DataFrame()

# Load and clean training and test sets
train_features_df = build_features_from_partition(train_subfolder)
test_features_df = build_features_from_partition(test_subfolder)

print(f"Loaded train features: {train_features_df.shape}")
print(f"Loaded test features: {test_features_df.shape}")

if not train_features_df.empty:
    display(train_features_df.head())


Using Train folder: /content/dataset/Student folder/train
Using Test folder: /content/dataset/Student folder/student_test
Loaded train features: (208417, 4)
Loaded test features: (19160, 4)


,timestamp,lat,lon,trip_id
0,1970-01-01 00:02:17.236968,36.695646,1.0,00000037
1,1970-01-01 00:02:17.236776,36.695580,2.0,00000037
2,1970-01-01 00:02:17.236816,36.695579,3.0,00000037
3,1970-01-01 00:02:17.236707,36.695527,4.0,00000037
4,1970-01-01 00:02:17.236694,36.695511,5.0,00000037


In [21]:
from google.colab import files

# Save your DataFrame first
train_features_df.to_csv('train_features.csv', index=False)
test_features_df.to_csv('test_features.csv', index=False)

# Trigger the browser download prompt for each file
files.download('train_features.csv')
files.download('test_features.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
import os
import shutil
from google.colab import files

# Let's check which folders exist so we don't hit a FileNotFoundError
available_folders = [f for f in os.listdir('/content') if os.path.isdir(os.path.join('/content', f)) and not f.startswith('.')]
print("Available directories in /content:", available_folders)

# Let's default to zipping the 'dataset' folder, or update to whatever folder you need!
folder_to_zip = '/content/dataset'
output_filename = '/content/dataset_bundle'

if os.path.exists(folder_to_zip):
    shutil.make_archive(output_filename, 'zip', folder_to_zip)
    print(f"Folder '{folder_to_zip}' successfully zipped to '{output_filename}.zip'!")
    # Trigger the download for the zip file
    files.download(f"{output_filename}.zip")
else:
    print(f"Error: The folder '{folder_to_zip}' does not exist. Please update 'folder_to_zip' to one of the available folders: {available_folders}")

Available directories in /content: ['dataset', 'drive', 'sample_data']
Folder '/content/dataset' successfully zipped to '/content/dataset_bundle.zip'!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>